# 跨靶点臂 MM-PBSA —— 36 条轨迹

> **本 notebook 由 `make_cross_notebooks.py` 从主臂 V3 版定点替换生成，非重写。**
> 与主臂版的全部差异即该脚本的替换表，每一处均断言命中次数。
> **白名单 `ALLOWED_ARMS` 已刻意加入 `cross_solv_v3`** —— 原白名单会正确地拒绝本臂；
> 扩充它是 2026-10-19 的一次决定，须随本 notebook 一同入档。
> 主臂产物（`step7_solv_v3` / `step6e_solv_v3` / `*_SOLV_V3.csv`）不受本notebook 影响。

---

# phase3 · MM-PBSA（跨靶点臂 cross_solv_v3）

在 PBC 校正后的 `md_fit.xtc` 上算 MM-PBSA → `dG_PBSA_mean` / `sigma_PBSA`（COMPASS 两轴）。

**相对 V2 的改动**

| | V2 | V3 |
|---|---|---|
| `ARMS` | `md_solv`, `md_cross_solv` | `cross_solv_v3` |
| `-cg` 组号 | 硬编码 `1 13` + `assert g[13]=='UNL'` | **按名解析**，并核验跨轨迹一致 |
| 输出根 | `results/step7_solv/` | `results/step7_cross_v3/`（不覆盖 V2） |
| 拓扑复制 | `cp $SRC/*.top $SRC/*.itp` | 另需 `cp -r ligand.acpype/`（V3 的配体 itp 在子目录） |

> **一个必须显式记下的巧合**：V3 的配体残基组 `MOL` **恰好也落在组号 13**，与 V2 的 `UNL` 相同。
> 若沿用 `-cg 1 13`，它会**纯属巧合地**指向正确的组；而 `assert g[13]=='UNL'` 会失败，
> 且失败信息与真实原因无关。按名解析同时消除两者。

**帧窗口无须改动**：V3 的 `nstxout-compressed = 5000` + 100 ns 同样给出 10,001 帧，
`step7_pbsa_fix_v4.in` 的 `startframe=5001, endframe=10000, interval=25` 为后 50 ns（200 帧，间隔 250 ps）。

**安全**：输出写新目录 `results/step7_cross_v3/`；不动 `md.xtc` / `md_fit.xtc`；不触及 V2 结果。

In [8]:
# === 版本闸：确认这是 V3 notebook，且不会误触 V2 ===
from pathlib import Path
import sys

EXPECTED_ARM = "cross_solv_v3"

print("=" * 62)
print("  这是 V3 版 notebook —— 只处理 md_solv_v3，输出全部带 _v3 / _V3")
print("  V2 版（md_solv / md_cross_solv）是另一个文件，不要在同一会话中打开")
print("=" * 62)

_root = Path("/mnt/hpc/home/wanghongmei/lang2")
assert (_root / EXPECTED_ARM).is_dir(), f"缺 {EXPECTED_ARM}"
_n = len(list((_root / EXPECTED_ARM).glob("*/*/06_md/md_fit.xtc")))
print(f"{EXPECTED_ARM}: {_n} 条 md_fit.xtc （应为 36）")
assert _n == 36, f"md_fit.xtc 数量 {_n} != 36 —— phase2 跨臂是否已完成？"   # 36 的依据：AMD-2 2026-10-31

  这是 V3 版 notebook —— 只处理 md_solv_v3，输出全部带 _v3 / _V3
  V2 版（md_solv / md_cross_solv）是另一个文件，不要在同一会话中打开
cross_solv_v3: 36 条 md_fit.xtc （应为 36）


In [9]:
# === 配置 ===
from pathlib import Path
import os, glob, re, shutil, json
import pandas as pd, numpy as np

ROOT  = Path("/mnt/hpc/home/wanghongmei/lang2")
GMXRC = "/mnt/hpc/home/wanghongmei/apps/gromacs-2025.4-cuda12.1/bin/GMXRC"
ARMS  = ["cross_solv_v3"]
N_EXPECT = 36   # 依据 AMD-2 2026-10-31：该臂以两批达到 36（30 + 6）

# gmx_MMPBSA 输入（PB-only, radiopt=1）—— 自动定位，不写死路径
_cands = [ROOT/"release/PRISM_Dyn/core/step7_pbsa_fix_v4.in",
          ROOT/"core/step7_pbsa_fix_v4.in",
          ROOT/"scripts/step7_pbsa_fix_v4.in"]
MMPBSA_IN = next((p for p in _cands if p.is_file()), None)
assert MMPBSA_IN is not None, f"未找到 step7_pbsa_fix_v4.in，已查: {[str(p) for p in _cands]}"

OUT_ROOT = ROOT / "results" / "step7_cross_v3"         # 新输出根，不覆盖主臂
TRAJ     = "md_fit.xtc"

SBATCH = dict(partition="gpu", account="hongmeilab", qos="normal",
              cpus=8, time="06:00:00", gres="")
CONCURRENCY = 20

ALLOWED_ARMS = {"md_solv", "md_cross_solv", "md_solv_v3", "md_cross_solv_v3",
                "cross_solv_v3"}        # ← 跨靶点臂目录，2026-10-19 刻意加入
assert set(ARMS) <= ALLOWED_ARMS and not (set(ARMS) & {"md", "md_cross"})

print("mmpbsa.in :", MMPBSA_IN)
print(MMPBSA_IN.read_text())          # 打印帧窗口，确认 5001-10000 interval 25（200 帧）—— 与主臂 99 条同一取样
print("输出根    :", OUT_ROOT)

mmpbsa.in : /mnt/hpc/home/wanghongmei/lang2/release/PRISM_Dyn/core/step7_pbsa_fix_v4.in
# ─────────────────────────────────────────────────────────
# PBSA 修复 v4 — fillratio=2.0 + radiopt=1 (双重修复)
#
# 同时解决：
#   1. 不兼容原子类型 (radiopt=1)
#   2. 内存压力 (fillratio=2.0)
# ─────────────────────────────────────────────────────────
&general
  startframe=5001,
  endframe=10000,
  interval=25,
  verbose=2,
  keep_files=0,
/
&pb
  istrng=0.150,
  fillratio=2.0,
  inp=2,
  radiopt=1,
/

输出根    : /mnt/hpc/home/wanghongmei/lang2/results/step7_cross_v3


In [10]:
# === 按名解析 -cg 组号（替代 V2 的硬编码 1 13）===
import re, glob
def ndx_groups(ndx):
    return [m.group(1) for m in
            (re.match(r'\s*\[\s*(.+?)\s*\]', ln) for ln in open(ndx)) if m]

def lig_group(ndx, candidates=("MOL", "UNL")):
    g = ndx_groups(ndx)
    hit = [c for c in candidates if c in g]
    if len(hit) != 1:
        raise AssertionError(f"{ndx}: 配体组不唯一或缺失，命中 {hit}；现有组 {g}")
    return hit[0], g.index(hit[0])

def rec_group(ndx, name="Protein"):
    g = ndx_groups(ndx)
    assert g.count(name) == 1, f"{ndx}: '{name}' 组不唯一"
    return g.index(name)

ndxs = sorted(glob.glob(f"{ROOT}/{ARMS[0]}/*/*/06_md/index.ndx"))
assert ndxs, f"未找到 index.ndx —— {ARMS[0]} 是否已产出轨迹？"

LIG_NAME, LIG_NUM = lig_group(ndxs[0])
REC_NUM = rec_group(ndxs[0], "Protein")
print(f"✓ 配体组 = {LIG_NAME} (组 {LIG_NUM}) | 受体组 = Protein (组 {REC_NUM})")
if LIG_NUM == 13:
    print("  注: 组号恰为 13，与 V2 的 UNL 相同 —— 这是巧合，本 notebook 不依赖它")

# 跨轨迹一致性：runner 用单一 -cg，故组号必须在全部轨迹上相同
# 跨臂横跨两个受体，组号未必一致。先按受体打印，断言失败时输出即是修法。
from collections import defaultdict
by_rec = defaultdict(set)
for n in ndxs:
    rec = "3V3M" if "3V3M" in n else ("4YLU" if "4YLU" in n else "?")
    by_rec[rec].add((lig_group(n), rec_group(n, "Protein")))
for rec in sorted(by_rec):
    print(f"  {rec}: (配体, 受体) 组号 = {by_rec[rec]}   ({sum(1 for n in ndxs if rec in n)} 条)")

allg = {(lig_group(n), rec_group(n, "Protein")) for n in ndxs}
assert len(allg) == 1, (
    f"组号在轨迹间不一致: {allg}\n"
    "→ 两受体组号不同。修法：按受体拆 manifest，各自用自己的 -cg 提交两个数组作业。\n"
    "  不要改成取第一条的组号 —— 那会让另一受体算在错误的组上，且不报错。")
print(f"✓ 组号在全部 {len(ndxs)} 条轨迹上一致")

# 配体原子数核验 —— **不与冻结记录对账**。
# 实测 2026-10-19：跨臂的两份记录都不含原子数 ——
#   Node05_prep_freeze_cross.json  (module=drv_cross_arm) 无 invariants 块；
#   Node05_prep_verify_cross.json  (module=drv_cross_verify) 每体系只存
#   {all_passed, system, target} 三个字段，核验结论在、数值不在。
# 故改为两条**由数据自身可判定**的检查，不引用任何记录里的数字。
n_lig, meta = {}, {}
for n in ndxs:
    sysname = Path(n).parents[2].name            # pool_mol_XXX__REC__pose
    txt = open(n).read().split(f"[ {LIG_NAME} ]")[1].split("[")[0]
    n_lig[sysname] = len(txt.split())
    parts = sysname.split("__")
    meta[sysname] = (parts[0], parts[1])

# ① 同一分子在两受体上的配体原子数必须相同 —— I2b 的可计算形式
per_mol = {}
for sysname, cnt in n_lig.items():
    mol, rec = meta[sysname]
    per_mol.setdefault(mol, {})[rec] = cnt
bad = {m: d for m, d in per_mol.items() if len(set(d.values())) != 1}
assert not bad, f"同一分子在两受体上配体原子数不一致: {bad}"

# ② 逐分子打印，供与 §B12.3 人工对照。**刻意不作断言** —— 把记录里的
#    数字写死进代码正是 D14 的陷阱；人看一眼即可，机器不该据此通过或失败。
print(f"✓ {len(n_lig)} 个体系；每分子在两受体上的配体原子数一致：")
for m in sorted(per_mol):
    print(f"    {m}: {sorted(per_mol[m].items())}")

✓ 配体组 = MOL (组 13) | 受体组 = Protein (组 1)
  注: 组号恰为 13，与 V2 的 UNL 相同 —— 这是巧合，本 notebook 不依赖它
  3V3M: (配体, 受体) 组号 = {(('MOL', 13), 1)}   (18 条)
  4YLU: (配体, 受体) 组号 = {(('MOL', 13), 1)}   (18 条)
✓ 组号在全部 36 条轨迹上一致
✓ 12 个体系；每分子在两受体上的配体原子数一致：
    pool_mol_001: [('3V3M', 55), ('4YLU', 55)]
    pool_mol_004: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_012: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_057: [('3V3M', 42), ('4YLU', 42)]
    pool_mol_074: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_279: [('3V3M', 54), ('4YLU', 54)]


In [11]:
# === 枚举待算（有 md_fit.xtc、尚无结果）===
def list_targets():
    todo, done, nofit = [], [], []
    have = set()
    for arm in ARMS:
        for fit in sorted(glob.glob(f"{ROOT}/{arm}/*/*/06_md/{TRAJ}")):
            d = Path(fit).parent
            if "_validate" in str(d): continue
            have.add(str(d))
            mol, rep = d.parents[1].name, d.parents[0].name
            res = OUT_ROOT / "pbsa" / mol / rep / "FINAL_RESULTS.dat"
            (done if res.exists() else todo).append(d)
        for gro in glob.glob(f"{ROOT}/{arm}/*/*/06_md/md.gro"):
            d = str(Path(gro).parent)
            if "_validate" not in d and d not in have: nofit.append(d)
    return todo, done, nofit

todo, done, nofit = list_targets()
print(f"待算: {len(todo)}   已完成: {len(done)}   缺 {TRAJ}: {len(nofit)}  "
      f"(PBC 出齐后待算应 = {N_EXPECT})")

待算: 0   已完成: 36   缺 md_fit.xtc: 0  (PBC 出齐后待算应 = 36)


In [6]:
# === 生成 SLURM 作业 ===
LOGDIR = ROOT / "logs" / "phase3_cross_v3"; LOGDIR.mkdir(parents=True, exist_ok=True)
MANI = ROOT / "_pbsa_cross_v3_manifest.txt"
MANI.write_text("\n".join(str(d) for d in todo) + "\n")

runner = f"""#!/usr/bin/env bash
set -uo pipefail
SRC="$1"                                   # 06_md 源目录
conda activate md_env 2>/dev/null || source activate md_env 2>/dev/null || true
set +u; source "{GMXRC}" >/dev/null 2>&1; set -u
MOL=$(basename "$(dirname "$(dirname "$SRC")")")     # pool_mol_XXX__3V3M-或-4YLU__primary
REP=$(basename "$(dirname "$SRC")")                  # r1/r2/r3
WORK="{OUT_ROOT}/pbsa/$MOL/$REP"
[ -f "$WORK/FINAL_RESULTS.dat" ] && {{ echo "SKIP(已算) $MOL/$REP"; exit 0; }}
[ -f "$SRC/{TRAJ}" ] || {{ echo "无 {TRAJ}: $SRC"; exit 1; }}
mkdir -p "$WORK"; cd "$WORK"
# 本地拓扑（解析 #include）。V3 的配体 itp 在子目录 ligand.acpype/，须整目录复制。
cp "$SRC"/*.top ./ 2>/dev/null || true
cp -r "$SRC"/ligand.acpype ./ 2>/dev/null || true
cp "$SRC"/../05_npt/posre_prot.itp ./ 2>/dev/null || true
echo "[PBSA] $MOL/$REP $(date -Is)"
gmx_MMPBSA -O -i "{MMPBSA_IN}" -cs "$SRC/md.tpr" -ct "$SRC/{TRAJ}" \\
  -ci "$SRC/index.ndx" -cp topol.top -cg {REC_NUM} {LIG_NUM} \\
  -o FINAL_RESULTS.dat -eo FINAL_RESULTS.csv -nogui >> run.log 2>&1
echo "[PBSA-DONE] $MOL/$REP rc=$? $(date -Is)"
"""
RUNNER = ROOT / "phase3_cross_v3_pbsa_one.sh"; RUNNER.write_text(runner); os.chmod(RUNNER, 0o755)

gres = f"#SBATCH --gres={SBATCH['gres']}\n" if SBATCH["gres"] else ""
sb = f"""#!/usr/bin/env bash
#SBATCH --job-name=phase3_cross_v3_pbsa
#SBATCH --output={LOGDIR}/pbsa_%A_%a.out
#SBATCH --error={LOGDIR}/pbsa_%A_%a.err
#SBATCH --array=1-{max(len(todo),1)}%{CONCURRENCY}
#SBATCH --partition={SBATCH['partition']}
#SBATCH --account={SBATCH['account']}
#SBATCH --qos={SBATCH['qos']}
#SBATCH --cpus-per-task={SBATCH['cpus']}
#SBATCH --time={SBATCH['time']}
{gres}d=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" {MANI})
[ -z "$d" ] && exit 0
bash {RUNNER} "$d"
"""
SB = ROOT / "phase3_cross_v3_pbsa.sbatch"; SB.write_text(sb)
print("已写:", MANI.name, RUNNER.name, SB.name)
print("日志:", LOGDIR)
print(f"提交: sbatch {SB}   (1-{len(todo)}%{CONCURRENCY})")
print("  （上为绝对路径，在任何工作目录下提交都有效）")
print(f"使用 -cg {REC_NUM} {LIG_NUM}  ({LIG_NAME})")

已写: _pbsa_cross_v3_manifest.txt phase3_cross_v3_pbsa_one.sh phase3_cross_v3_pbsa.sbatch
日志: /mnt/hpc/home/wanghongmei/lang2/logs/phase3_cross_v3
提交: sbatch /mnt/hpc/home/wanghongmei/lang2/phase3_cross_v3_pbsa.sbatch   (1-6%20)
  （上为绝对路径，在任何工作目录下提交都有效）
使用 -cg 1 13  (MOL)


In [12]:
# === DELTA-TOTAL 解析器（读 .dat 的 Delta 段 TOTAL；V2 BUG#2 修正沿用）===
def parse_delta_total(dat_path, method="POISSON"):
    txt = open(dat_path, errors="ignore").read()
    if method == "GB":
        seg = txt.split("GENERALIZED BORN")[-1].split("POISSON BOLTZMANN")[0]
    else:
        seg = txt.split("POISSON BOLTZMANN")[-1] if "POISSON BOLTZMANN" in txt else txt
    if "Delta" in seg:
        for ln in seg.split("Delta")[-1].splitlines():
            if re.match(r"\s*(Δ?TOTAL|TOTAL)\b", ln):
                nums = re.findall(r"-?\d+\.\d+", ln)
                if nums: return float(nums[0])
    return np.nan
print("parse_delta_total() 已定义")

parse_delta_total() 已定义


In [13]:
# === 收集 + 聚合（SLURM 全部完成后跑）===
rows = []
for dat in glob.glob(f"{OUT_ROOT}/pbsa/*/*/FINAL_RESULTS.dat"):
    d = Path(dat)
    rows.append(dict(mol_dir=d.parent.parent.name, rep=d.parent.name,
                     dG_PB=parse_delta_total(dat, "POISSON")))
per = pd.DataFrame(rows)
print(f"解析到 {len(per)} 条  (目标 {N_EXPECT})")
if len(per):
    print(per.head(10).to_string(index=False))
    bad = per[(per.dG_PB < -200) | (per.dG_PB > 50) | per.dG_PB.isna()]
    if len(bad):
        print(f"\n⚠ {len(bad)} 条数值离奇（应 ~ -10 至 -40 kcal/mol），先查解析器再聚合：")
        print(bad.head().to_string(index=False))
    agg = per.groupby("mol_dir").agg(dG_PBSA_mean=("dG_PB", "mean"),
                                     sigma_PBSA=("dG_PB", "std"),
                                     n_rep=("dG_PB", "size")).reset_index()
    agg["mol_id"] = agg.mol_dir.str.split("__").str[0]
    agg["target"] = agg.mol_dir.str.split("__").str[1]
    out = ROOT / "results" / "phase3_cross_dG_aggregated_SOLV_V3.csv"
    agg.to_csv(out, index=False)
    print("\n已存", out.name); print(agg.head(12).to_string(index=False))

解析到 36 条  (目标 36)
                    mol_dir rep  dG_PB
pool_mol_004__4YLU__primary  r3  -2.78
pool_mol_004__4YLU__primary  r2   3.06
pool_mol_004__4YLU__primary  r1  -2.08
pool_mol_012__3V3M__primary  r3  -4.54
pool_mol_012__3V3M__primary  r2  -7.96
pool_mol_012__3V3M__primary  r1  -6.82
pool_mol_012__4YLU__primary  r3   1.57
pool_mol_012__4YLU__primary  r2   5.16
pool_mol_012__4YLU__primary  r1   6.20
pool_mol_057__4YLU__primary  r3  -9.15

已存 phase3_cross_dG_aggregated_SOLV_V3.csv
                    mol_dir  dG_PBSA_mean  sigma_PBSA  n_rep       mol_id target
pool_mol_001__3V3M__primary      2.083333    3.353570      3 pool_mol_001   3V3M
pool_mol_001__4YLU__primary      4.223333    2.144116      3 pool_mol_001   4YLU
pool_mol_004__3V3M__primary      0.983333    0.696730      3 pool_mol_004   3V3M
pool_mol_004__4YLU__primary     -0.600000    3.188918      3 pool_mol_004   4YLU
pool_mol_012__3V3M__primary     -6.440000    1.741379      3 pool_mol_012   3V3M
pool_mol_012__4YLU__prim

## 顺序
1. phase2 出齐 36 条 `md_fit.xtc` → 本 notebook「待算」= 36。（99 是自主臂复制而来的残留；在主臂 99 是对的，故只在跨臂改。）
2. 组号解析 cell 必须先通过（含与 Node 05 I2b 的配体原子数对账）。
3. `sbatch phase3_cross_v3_pbsa.sbatch` → `results/step7_cross_v3/pbsa/<mol>/<rep>/`。
4. 收集 cell → `results/phase3_cross_dG_aggregated_SOLV_V3.csv`。

## 首条跑完后必须核对
`results/step7_cross_v3/pbsa/<第一个>/r1/FINAL_RESULTS.dat` 中 Delta 段的 TOTAL 行，
确认解析器取到的是 **−10 ~ −40 kcal/mol** 量级，而非 ~−2600（V2 曾因对多块 CSV
整列平均而得到后者）。收集 cell 已内置该量级检查。